# What Is Data Governance? A Data Engineer's Version (YouTube T1-01 Demo)

| Metadata | Specification |
| :--- | :--- |
| **Target Persona** | Data Engineers, Analytics Engineers, Platform Engineers |
| **Difficulty Level** | Level 100 (L100 — Foundational Engineering Walkthrough) |
| **Prerequisites** | Google Cloud Project with BigQuery, Cloud Storage, and Knowledge Catalog APIs enabled |
| **Estimated Time** | 10 minutes |

---

## 1. The 2 AM Silent Data Drop Story (Cold Open)

It is 2:00 AM. Your phone rings. The executive sales dashboard (`sales_mart.daily_revenue`) is wrong—revenue dropped by 50% overnight, and the executive leadership meeting starts at 9:00 AM.

You open your data orchestration dashboard. **Every single job is green.** No errors, no failed tasks, nothing crashed.

So what happened? Yesterday, an upstream application team stopped emitting one specific order event type. The table schema did not change, and the SQL transformation ran without syntax or runtime errors—it simply processed half the rows silently. Nothing broke; something quietly went missing. Before 9:00 AM, you must answer three operational questions:
1. **Where did this number come from?**
2. **Which upstream table caused the drop?**
3. **Who owns that upstream table so you can fix it?**

### Data Swamp vs. Data With Context: The 4 Engineering Questions

When most engineers hear "data governance," they picture a 50-page PDF policy document that nobody reads and meetings nobody wants to attend. From a pure data engineering perspective, governance is not bureaucracy—it is how your data platform automatically answers four engineering questions:

1. **Discovery**: What data do we have across BigQuery and Cloud Storage, and where is it?
2. **Quality**: Can I trust the numbers inside this table right now?
3. **Lineage**: Where did this data come from, and what breaks downstream if I touch it?
4. **Access & Ownership**: Who owns that table so you can contact them at 2 AM, and who is allowed to read sensitive columns?

In a **Data Swamp**, your storage works and your compute is fast, but nobody knows what anything means. You see three tables named `customer_orders_final` across different datasets and Cloud Storage buckets, and if you have to ask five teams in Slack which one is authoritative, you do not have governance.

In a governed platform (**Data With Context**), we do not move a single byte of data or rewrite our pipelines. Instead, **Knowledge Catalog** enriches every asset with four operational signals: what it is, whether it is healthy, where it came from, and who owns it.

## 2. Environment Configuration & Parameter Validation

Configure your Google Cloud project parameters below. The parameter cell enforces fail-fast validation to ensure placeholder strings are replaced before interacting with live Google Cloud APIs.

> ℹ️ **Note**: In Knowledge Catalog, BigQuery datasets provisioned in the `US` multi-region map to the catalog control plane location `us` (`projects/{PROJECT_ID}/locations/us/entryGroups/@bigquery`), whereas regional Cloud Storage filesets map to their regional compute endpoint (such as `us-central1`).

In [ ]:
# Configure Google Cloud environment parameters and fail-fast validation
PROJECT_ID = "your-project-id"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "your-project-id":
    raise ValueError("Please set PROJECT_ID to your active Google Cloud project ID before running.")

# Fixed tutorial constants (do not require user modification)
LOCATION = "us-central1"
BQ_LOCATION = "US"
DATASET_ID = "sales_mart"
OWNER_EMAIL = "data-eng-oncall@example.com"

BUCKET_NAME = f"{PROJECT_ID}-t101-swamp-demo"
CATALOG_BQ_LOCATION = BQ_LOCATION.lower()

print(f"Configured Project ID          : {PROJECT_ID}")
print(f"Regional Compute/Storage       : {LOCATION}")
print(f"BigQuery Multi-Region Location : {BQ_LOCATION} (Catalog Location: {CATALOG_BQ_LOCATION})")
print(f"Target BigQuery Dataset        : {DATASET_ID}")
print(f"Target Cloud Storage Bucket    : gs://{BUCKET_NAME}")
print(f"Designated On-Call Owner Email : {OWNER_EMAIL}")

## 3. Install & Import Native SDK Clients

We verify package availability via `importlib.util.find_spec` before invoking `subprocess.check_call` so PEP 668 externally managed Python environments succeed cleanly. We then initialize the native Google Cloud client libraries for **BigQuery**, **Cloud Storage**, **Knowledge Catalog**, and **Data Lineage**.

In [ ]:
# Check find_spec before installing dependencies and initialize native Google Cloud SDK clients
import importlib.util
import subprocess
import sys
import time
from pathlib import Path

REQUIRED_PACKAGES = [
    ("google.cloud.dataplex_v1", "google-cloud-dataplex"),
    ("google.cloud.bigquery", "google-cloud-bigquery"),
    ("google.cloud.storage", "google-cloud-storage"),
    ("google.cloud.datacatalog_lineage_v1", "google-cloud-datacatalog-lineage"),
    ("tabulate", "tabulate"),
]

for module_name, pip_package in REQUIRED_PACKAGES:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing package: {pip_package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pip_package])

from google.api_core.exceptions import (
    AlreadyExists,
    InternalServerError,
    NotFound,
    ResourceExhausted,
    ServiceUnavailable,
)
from google.cloud import bigquery
from google.cloud import datacatalog_lineage_v1 as lineage_v1
from google.cloud import dataplex_v1
from google.cloud import storage
from google.protobuf import timestamp_pb2
from tabulate import tabulate

bq_client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
storage_client = storage.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()
lineage_client = lineage_v1.LineageClient()

print("✓ Initialized native BigQuery, Cloud Storage, Knowledge Catalog, and Data Lineage clients.")

## 4. Pre-Flight Demo Setup: Provisioning BigQuery Tables, Cloud Storage Files, and SQL Lineage

To make this demo 100% self-contained, we implement the pre-flight setup checklist from Section 2.1 of the episode runbook:

1. **BigQuery Dataset & Upstream Source Table**: Create dataset `sales_mart` (location `US`) and table `sales_mart.customer_orders` populated with sample customer checkout events.
2. **Cloud Storage Archive & Knowledge Catalog Fileset**: Create bucket `gs://{PROJECT_ID}-t101-swamp-demo`, upload a raw CSV file (`customer_orders_archive.csv`), and register a custom **Knowledge Catalog** Fileset Entry (`customer_orders_archive_fileset`) inside EntryGroup `t101_storage_group`. This allows a single search query for `"customer"` to discover both structured BigQuery tables and raw Cloud Storage files.
3. **Downstream Transformation SQL**: Execute the aggregation query that builds the executive dashboard table `sales_mart.daily_revenue` from `sales_mart.customer_orders`:
   ```sql
   CREATE OR REPLACE TABLE sales_mart.daily_revenue AS
   SELECT
     DATE(order_timestamp) AS revenue_date,
     COUNT(*) AS order_count,
     SUM(order_amount) AS total_revenue
   FROM sales_mart.customer_orders
   GROUP BY revenue_date
   ```

> ⚠️ **Important**: BigQuery Data Lineage ingestion runs asynchronously in the background and typically requires 15 to 30 minutes (up to 24 hours under official SLA) after query execution to populate the visual graph in the Google Cloud console UI. Table schemas and catalog entries, however, are available immediately. To ensure deterministic programmatic lineage verification in this notebook without waiting 30 minutes, we also record the explicit SQL transformation process and lineage event via `LineageClient`.

In [ ]:
# 1. Create BigQuery dataset sales_mart and upstream table customer_orders
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = BQ_LOCATION
bq_client.create_dataset(dataset_ref, exists_ok=True)
print(f"✓ Verified BigQuery dataset: {PROJECT_ID}.{DATASET_ID} (Location: {BQ_LOCATION})")

create_orders_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.customer_orders` AS
SELECT
  'ORD-1001' AS order_id,
  'CUST-501' AS customer_id,
  'CHECKOUT_COMPLETED' AS event_type,
  250.00 AS order_amount,
  TIMESTAMP('2026-09-15 14:20:00 UTC') AS order_timestamp
UNION ALL
SELECT
  'ORD-1002' AS order_id,
  'CUST-502' AS customer_id,
  'CHECKOUT_COMPLETED' AS event_type,
  400.00 AS order_amount,
  TIMESTAMP('2026-09-15 18:45:00 UTC') AS order_timestamp
UNION ALL
SELECT
  'ORD-1003' AS order_id,
  'CUST-503' AS customer_id,
  'CHECKOUT_COMPLETED' AS event_type,
  350.00 AS order_amount,
  TIMESTAMP('2026-09-16 01:15:00 UTC') AS order_timestamp
"""
bq_client.query(create_orders_sql).result()
print(f"✓ Created upstream table: {PROJECT_ID}.{DATASET_ID}.customer_orders (3 sample rows)")

# 2. Provision Cloud Storage bucket, upload CSV archive, and register Knowledge Catalog Fileset Entry
bucket_obj = storage_client.bucket(BUCKET_NAME)
if not bucket_obj.exists():
    storage_client.create_bucket(bucket_obj, location=LOCATION)
    print(f"✓ Created Cloud Storage bucket: gs://{BUCKET_NAME}")
else:
    print(f"✓ Cloud Storage bucket already exists: gs://{BUCKET_NAME}")

csv_filename = "customer_orders_archive.csv"
csv_content = (
    "order_id,customer_id,event_type,order_amount,order_timestamp\n"
    "ORD-0900,CUST-101,CHECKOUT_COMPLETED,150.00,2026-09-01T10:00:00Z\n"
    "ORD-0901,CUST-102,CHECKOUT_COMPLETED,275.50,2026-09-01T11:30:00Z\n"
)
blob = bucket_obj.blob(csv_filename)
blob.upload_from_string(csv_content, content_type="text/csv")
gcs_archive_uri = f"gs://{BUCKET_NAME}/{csv_filename}"
print(f"✓ Uploaded Cloud Storage CSV archive: {gcs_archive_uri}")

regional_parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"
ENTRY_GROUP_ID = "t101_storage_group"
ENTRY_TYPE_ID = "t101-fileset-type"
FILESET_ENTRY_ID = "customer_orders_archive_fileset"

entry_group_name = f"{regional_parent}/entryGroups/{ENTRY_GROUP_ID}"
entry_type_name = f"{regional_parent}/entryTypes/{ENTRY_TYPE_ID}"
fileset_entry_name = f"{entry_group_name}/entries/{FILESET_ENTRY_ID}"

try:
    eg_op = catalog_client.create_entry_group(
        parent=regional_parent,
        entry_group_id=ENTRY_GROUP_ID,
        entry_group=dataplex_v1.EntryGroup(
            name=entry_group_name,
            display_name="T1-01 Cloud Storage Swamp Demo Group",
            description="Entry group holding Cloud Storage filesets for unified discovery demo.",
        ),
    )
    eg_op.result()
    print(f"✓ Created Knowledge Catalog EntryGroup: {entry_group_name}")
except AlreadyExists:
    print(f"✓ EntryGroup already exists: {entry_group_name}")

try:
    et_op = catalog_client.create_entry_type(
        parent=regional_parent,
        entry_type_id=ENTRY_TYPE_ID,
        entry_type=dataplex_v1.EntryType(
            name=entry_type_name,
            display_name="Cloud Storage CSV Fileset",
            description="Custom entry type representing archived CSV files in Cloud Storage.",
        ),
    )
    et_op.result()
    print(f"✓ Created Knowledge Catalog EntryType: {entry_type_name}")
except AlreadyExists:
    print(f"✓ EntryType already exists: {entry_type_name}")

fileset_entry_spec = dataplex_v1.Entry(
    name=fileset_entry_name,
    entry_type=entry_type_name,
    fully_qualified_name=f"gcs:{BUCKET_NAME}.customer_orders_archive",
    entry_source=dataplex_v1.EntrySource(
        resource=gcs_archive_uri,
        system="Cloud Storage",
        platform="Google Cloud",
        display_name="customer_orders_archive_fileset",
        description="Historical customer orders CSV archive stored in Cloud Storage bucket.",
        location=LOCATION,
    ),
)

try:
    catalog_client.create_entry(
        parent=entry_group_name,
        entry_id=FILESET_ENTRY_ID,
        entry=fileset_entry_spec,
    )
    print(f"✓ Registered Cloud Storage Fileset Entry: {fileset_entry_name}")
except AlreadyExists:
    catalog_client.update_entry(request=dataplex_v1.UpdateEntryRequest(entry=fileset_entry_spec))
    print(f"✓ Updated existing Cloud Storage Fileset Entry: {fileset_entry_name}")

# 3. Execute transformation SQL query to build sales_mart.daily_revenue
transformation_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.daily_revenue` AS
SELECT
  DATE(order_timestamp) AS revenue_date,
  COUNT(*) AS order_count,
  SUM(order_amount) AS total_revenue
FROM `{PROJECT_ID}.{DATASET_ID}.customer_orders`
GROUP BY revenue_date
"""
bq_client.query(transformation_sql).result()
print(f"✓ Executed SQL transformation: {PROJECT_ID}.{DATASET_ID}.customer_orders -> {PROJECT_ID}.{DATASET_ID}.daily_revenue")

# Record explicit Data Lineage Process/Run/Event for immediate programmatic verification
lineage_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_BQ_LOCATION}"
lineage_proc = lineage_client.create_process(
    parent=lineage_parent,
    process=lineage_v1.Process(
        display_name="bq_daily_revenue_aggregation_sql",
        attributes={"sql_query": transformation_sql.strip()},
    ),
)
lineage_process_name = lineage_proc.name

now_pb = timestamp_pb2.Timestamp()
now_pb.GetCurrentTime()
lineage_run = lineage_client.create_run(
    parent=lineage_process_name,
    run=lineage_v1.Run(
        display_name="nightly_2am_aggregation_run",
        start_time=now_pb,
        end_time=now_pb,
        state=lineage_v1.Run.State.COMPLETED,
    ),
)
lineage_event = lineage_client.create_lineage_event(
    parent=lineage_run.name,
    lineage_event=lineage_v1.LineageEvent(
        start_time=now_pb,
        end_time=now_pb,
        links=[
            lineage_v1.EventLink(
                source=lineage_v1.EntityReference(
                    fully_qualified_name=f"bigquery:{PROJECT_ID}.{DATASET_ID}.customer_orders"
                ),
                target=lineage_v1.EntityReference(
                    fully_qualified_name=f"bigquery:{PROJECT_ID}.{DATASET_ID}.daily_revenue"
                ),
            )
        ],
    ),
)
print(f"✓ Recorded Data Lineage transformation process: {lineage_process_name}")

## 5. Pre-Flight Demo Setup: Registering Table Ownership / Contacts in Knowledge Catalog

In Cut 3 of the video, after tracing the broken dashboard table (`sales_mart.daily_revenue`) back to its upstream SQL job, the engineer checks the table's **Contacts** / governance metadata to see who owns the pipeline at 2 AM without asking around in Slack.

Below, we provision a custom Level 100 AspectType (`t101-table-contacts`) in location `us` and attach it to the canonical **Knowledge Catalog** entry for `sales_mart.daily_revenue` (`projects/{PROJECT_ID}/locations/us/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/sales_mart/tables/daily_revenue`).

In [ ]:
# Provision custom L100 AspectType t101-table-contacts and attach ownership metadata to daily_revenue
CONTACTS_ASPECT_TYPE_ID = "t101-table-contacts"
bq_catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_BQ_LOCATION}"
contacts_aspect_type_name = f"{bq_catalog_parent}/aspectTypes/{CONTACTS_ASPECT_TYPE_ID}"

contacts_aspect_template = {
    "name": "T101TableContacts",
    "type": "record",
    "record_fields": [
        {
            "name": "data_owner",
            "type": "string",
            "index": 1,
            "annotations": {"description": "Designated on-call data engineer email address."},
        },
        {
            "name": "oncall_slack_channel",
            "type": "string",
            "index": 2,
            "annotations": {"description": "Primary Slack channel for 2 AM incident escalation."},
        },
        {
            "name": "stewardship_tier",
            "type": "string",
            "index": 3,
            "annotations": {"description": "Executive dashboard governance SLA tier."},
        },
    ],
}

try:
    at_op = catalog_client.create_aspect_type(
        parent=bq_catalog_parent,
        aspect_type_id=CONTACTS_ASPECT_TYPE_ID,
        aspect_type=dataplex_v1.AspectType(
            name=contacts_aspect_type_name,
            display_name="T1-01 Table Contacts & Stewardship",
            description="Level 100 governance aspect storing on-call data owner and escalation channel.",
            metadata_template=contacts_aspect_template,
        ),
    )
    at_op.result()
    print(f"✓ Created AspectType: {contacts_aspect_type_name}")
except AlreadyExists:
    print(f"✓ AspectType already exists: {contacts_aspect_type_name}")

# Poll for the canonical @bigquery entry for sales_mart.daily_revenue
daily_revenue_entry_name = (
    f"projects/{PROJECT_ID}/locations/{CATALOG_BQ_LOCATION}/entryGroups/@bigquery/entries/"
    f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/daily_revenue"
)

canonical_entry = None
for attempt in range(12):
    try:
        canonical_entry = catalog_client.get_entry(name=daily_revenue_entry_name)
        print(f"✓ Synchronized BigQuery catalog entry on attempt {attempt + 1}: {canonical_entry.name}")
        break
    except NotFound:
        print(f"  [Attempt {attempt + 1}/12] Waiting for BigQuery catalog auto-discovery sync...")
        time.sleep(3)

if canonical_entry is None:
    raise RuntimeError(f"BigQuery catalog entry not found after polling: {daily_revenue_entry_name}")

aspect_key = f"{PROJECT_ID}.{CATALOG_BQ_LOCATION}.{CONTACTS_ASPECT_TYPE_ID}"
updated_entry_payload = dataplex_v1.Entry(
    name=daily_revenue_entry_name,
    aspects={
        aspect_key: dataplex_v1.Aspect(
            aspect_type=contacts_aspect_type_name,
            data={
                "data_owner": OWNER_EMAIL,
                "oncall_slack_channel": "#revenue-data-oncall",
                "stewardship_tier": "TIER_1_EXECUTIVE_DASHBOARD",
            },
        )
    },
)

catalog_client.update_entry(
    request=dataplex_v1.UpdateEntryRequest(
        entry=updated_entry_payload,
        update_mask={"paths": ["aspects"]},
        aspect_keys=[aspect_key],
    )
)
print(f"✓ Attached '{CONTACTS_ASPECT_TYPE_ID}' aspect to {DATASET_ID}.daily_revenue (Owner: {OWNER_EMAIL})")

## 6. Cut 3 Console Demo 1: Question 1 — Unified Discovery Across BigQuery & Cloud Storage

In Cut 3 of the YouTube episode, we answer **Question 1 (Discovery)**: *"What data do we have, and where is it?"*

Instead of asking five teams in Slack where customer order data lives, we execute a single search query (`query="customer"`) across **Knowledge Catalog** (`catalog_client.search_entries`). Notice how one search bar scans across completely different storage engines—returning both our relational **BigQuery** table (`sales_mart.customer_orders`) and our raw **Cloud Storage** CSV fileset (`customer_orders_archive_fileset`).

In [ ]:
# Execute unified discovery search across BigQuery and Cloud Storage using query="customer"
search_location = f"projects/{PROJECT_ID}/locations/global"
demo_assets_table = []

for poll_idx in range(10):
    try:
        search_pager = catalog_client.search_entries(
            request=dataplex_v1.SearchEntriesRequest(
                name=search_location,
                query="customer",
                scope=f"projects/{PROJECT_ID}",
                page_size=50,
            )
        )
        demo_assets_table = []
        found_bq = False
        found_gcs = False
        for item_idx, res_item in enumerate(search_pager):
            if item_idx >= 100:
                break
            entry_obj = res_item.dataplex_entry
            fqn_str = entry_obj.fully_qualified_name or entry_obj.name
            if f"{DATASET_ID}.customer_orders" in fqn_str or "customer_orders_archive" in fqn_str:
                sys_label = entry_obj.entry_source.system or "BIGQUERY"
                res_uri = entry_obj.entry_source.resource or fqn_str
                demo_assets_table.append([sys_label, entry_obj.name.split("/")[-1], fqn_str, res_uri])
                if "bigquery" in fqn_str.lower() or "BIGQUERY" in sys_label.upper():
                    found_bq = True
                if "gcs:" in fqn_str.lower() or "storage" in sys_label.lower():
                    found_gcs = True
        if found_bq and found_gcs:
            break
    except (InternalServerError, ServiceUnavailable, ResourceExhausted) as transient_err:
        print(f"  [Search Index Poll {poll_idx + 1}/10] Transient search gateway throttle ({type(transient_err).__name__}), retrying...")
    print(f"  [Search Index Poll {poll_idx + 1}/10] Waiting for unified search index propagation...")
    time.sleep(5)

if not demo_assets_table:
    raise RuntimeError("Unified discovery search returned no matching demo assets after polling.")

print(f"=== Unified Knowledge Catalog Discovery Results (Query: 'customer') ===")
print(
    tabulate(
        demo_assets_table,
        headers=["Storage System", "Catalog Entry ID", "Fully Qualified Name", "Physical Resource URI"],
        tablefmt="github",
    )
)

## 7. Cut 3 & Cut 4 Demo 2: Question 3 & Ownership — The 22-Line Standalone Python Inspection Script

In Cut 4 of the episode, we emphasize a core engineering principle: *"As engineers, we don't want to click around a web UI every day. We want to query this metadata directly from code."*

Below, we execute the exact 22-line standalone inspection script (`t1_01_inspect_table_context.py`) from Section 5 of the whitepaper to fetch the `sales_mart.daily_revenue` table entry, print its update timestamp, and extract the attached **Owner Contact (`data-eng-oncall@example.com`)** and Slack escalation channel (`#revenue-data-oncall`). We also verify that the standalone script file is saved to `snippets/t1_01_inspect_table_context.py`.

> 💡 **Tip**: Passing `view=dataplex_v1.EntryView.ALL` in `GetEntryRequest` ensures that Knowledge Catalog returns all custom governance aspects and contact metadata attached to the BigQuery table entry.

In [ ]:
# Execute the exact 22-line inspection workflow from Section 5 (t1_01_inspect_table_context.py)
INSPECT_ENTRY_GROUP_ID = "@bigquery"
INSPECT_ENTRY_ID = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/daily_revenue"
inspect_entry_name = (
    f"projects/{PROJECT_ID}/locations/{CATALOG_BQ_LOCATION}/"
    f"entryGroups/{INSPECT_ENTRY_GROUP_ID}/entries/{INSPECT_ENTRY_ID}"
)

# Fetch table entry with all governance aspects attached
inspected_entry = catalog_client.get_entry(
    request={"name": inspect_entry_name, "view": dataplex_v1.EntryView.ALL}
)
print(f"Table FQN : {inspected_entry.fully_qualified_name}")
print(f"Updated   : {inspected_entry.update_time}")
for aspect_key_str, aspect_val in inspected_entry.aspects.items():
    if "contacts" in aspect_key_str or "governance" in aspect_key_str:
        print(f"Aspect [{aspect_key_str}] -> {dict(aspect_val.data)}")

# Verify and write standalone 22-line script to snippets/t1_01_inspect_table_context.py
snippet_path = Path("/usr/local/google/home/hyunuklim/git/knowledge-catalog/snippets/t1_01_inspect_table_context.py")
snippet_path.parent.mkdir(parents=True, exist_ok=True)
snippet_lines = [
    "# snippets/t1_01_inspect_table_context.py",
    "from google.cloud import dataplex_v1",
    "",
    "# Replace with your Google Cloud project, catalog location ('us' for BQ US multi-region), and target table",
    'PROJECT_ID = "your-project-id"',
    'LOCATION = "us"',
    'ENTRY_GROUP_ID = "@bigquery"',
    'ENTRY_ID = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/sales_mart/tables/daily_revenue"',
    "",
    'if PROJECT_ID == "your-project-id":',
    '    raise ValueError("Please set your actual PROJECT_ID before running.")',
    "",
    "client = dataplex_v1.CatalogServiceClient()",
    'entry_name = f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/{ENTRY_GROUP_ID}/entries/{ENTRY_ID}"',
    "",
    "# Fetch table entry with all governance aspects attached",
    'entry = client.get_entry(request={"name": entry_name, "view": dataplex_v1.EntryView.ALL})',
    'print(f"Table FQN : {entry.fully_qualified_name}")',
    'print(f"Updated   : {entry.update_time}")',
    "for aspect_key, aspect_val in entry.aspects.items():",
    '    if "contacts" in aspect_key or "governance" in aspect_key:',
    '        print(f"Aspect [{aspect_key}] -> {dict(aspect_val.data)}")',
]
snippet_path.write_text("\n".join(snippet_lines) + "\n", encoding="utf-8")
print(f"\n✓ Verified standalone snippet ({len(snippet_lines)} lines): {snippet_path}")

## 8. Programmatic Lineage Check: Tracing Upstream Dependencies

To answer **Question 3 (Lineage)** (*"Where did this data come from, and what upstream table caused the drop?"*), we query the **Data Lineage API** (`LineageClient`).

Below, we execute `lineage_client.search_links` targeting `bigquery:{PROJECT_ID}.sales_mart.daily_revenue` and inspect the recorded transformation process (`bq_daily_revenue_aggregation_sql`) to display the upstream source table (`sales_mart.customer_orders`), the exact SQL transformation query, and the downstream target table.

In [ ]:
# Query Data Lineage API to trace upstream tables feeding sales_mart.daily_revenue
target_entity = lineage_v1.EntityReference(
    fully_qualified_name=f"bigquery:{PROJECT_ID}.{DATASET_ID}.daily_revenue"
)
indexed_links = list(
    lineage_client.search_links(
        request=lineage_v1.SearchLinksRequest(
            parent=f"projects/{PROJECT_ID}/locations/{CATALOG_BQ_LOCATION}",
            target=target_entity,
        )
    )
)

# Traverse the recorded transformation process and events for immediate deterministic verification
proc_obj = lineage_client.get_process(name=lineage_process_name)
sql_query_executed = proc_obj.attributes.get("sql_query", "N/A")

lineage_rows = []
for run_obj in lineage_client.list_runs(parent=lineage_process_name):
    for event_obj in lineage_client.list_lineage_events(parent=run_obj.name):
        for link_edge in event_obj.links:
            lineage_rows.append([
                link_edge.source.fully_qualified_name,
                proc_obj.display_name,
                link_edge.target.fully_qualified_name,
                run_obj.state.name if hasattr(run_obj.state, "name") else str(run_obj.state),
            ])

print(f"=== Programmatic Data Lineage Trace (Indexed search_links count: {len(indexed_links)}) ===")
print(
    tabulate(
        lineage_rows,
        headers=["Upstream Source Table FQN", "Transformation Job", "Downstream Target FQN", "Run State"],
        tablefmt="github",
    )
)
print(f"\nExecuted SQL Query Captured in Lineage Process:\n{sql_query_executed}")

## 9. Resilient Standalone Teardown

Clean up all resources provisioned during this demo in reverse dependency order:
1. Custom Data Lineage Process (`bq_daily_revenue_aggregation_sql`)
2. Custom Knowledge Catalog Fileset Entry (`customer_orders_archive_fileset`), EntryGroup (`t101_storage_group`), EntryType (`t101-fileset-type`), and AspectType (`t101-table-contacts`)
3. Cloud Storage Bucket (`gs://{PROJECT_ID}-t101-swamp-demo`)
4. BigQuery Dataset (`sales_mart`)

The teardown cell executes directly without bypass flags, using self-contained imports and module-level `in locals()` variable guards.

In [ ]:
# Resilient Standalone Teardown (Self-contained imports, module-level guards, strict exception handling)
from google.api_core.exceptions import NotFound, ResourceExhausted
from google.cloud import bigquery
from google.cloud import datacatalog_lineage_v1 as lineage_v1
from google.cloud import dataplex_v1
from google.cloud import storage

teardown_bq_client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
teardown_storage_client = storage.Client(project=PROJECT_ID)
teardown_catalog_client = dataplex_v1.CatalogServiceClient()
teardown_lineage_client = lineage_v1.LineageClient()

# 1. Delete Data Lineage Process
if "lineage_process_name" in locals() and lineage_process_name:
    try:
        teardown_lineage_client.delete_process(name=lineage_process_name)
        print(f"✓ Deleted Data Lineage process: {lineage_process_name}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"Lineage process already removed or rate-limited: {exc}")

# 2. Delete Custom Knowledge Catalog Fileset Entry, EntryGroup, EntryType, and AspectType
if "fileset_entry_name" in locals() and fileset_entry_name:
    try:
        teardown_catalog_client.delete_entry(name=fileset_entry_name)
        print(f"✓ Deleted Knowledge Catalog Fileset Entry: {fileset_entry_name}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"Fileset Entry already removed: {exc}")

if "entry_group_name" in locals() and entry_group_name:
    try:
        eg_del_op = teardown_catalog_client.delete_entry_group(name=entry_group_name)
        eg_del_op.result()
        print(f"✓ Deleted Knowledge Catalog EntryGroup: {entry_group_name}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"EntryGroup already removed: {exc}")

if "entry_type_name" in locals() and entry_type_name:
    try:
        et_del_op = teardown_catalog_client.delete_entry_type(name=entry_type_name)
        et_del_op.result()
        print(f"✓ Deleted Knowledge Catalog EntryType: {entry_type_name}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"EntryType already removed: {exc}")

if "contacts_aspect_type_name" in locals() and contacts_aspect_type_name:
    try:
        at_del_op = teardown_catalog_client.delete_aspect_type(name=contacts_aspect_type_name)
        at_del_op.result()
        print(f"✓ Deleted Knowledge Catalog AspectType: {contacts_aspect_type_name}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"AspectType already removed: {exc}")

# 3. Delete Cloud Storage Bucket and Objects
if "BUCKET_NAME" in locals() and BUCKET_NAME:
    try:
        cleanup_bucket = teardown_storage_client.bucket(BUCKET_NAME)
        if cleanup_bucket.exists():
            cleanup_bucket.delete(force=True)
            print(f"✓ Deleted Cloud Storage bucket and contents: gs://{BUCKET_NAME}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"Cloud Storage bucket already removed: {exc}")

# 4. Delete BigQuery Dataset sales_mart
if "DATASET_ID" in locals() and DATASET_ID:
    try:
        teardown_bq_client.delete_dataset(
            f"{PROJECT_ID}.{DATASET_ID}",
            delete_contents=True,
            not_found_ok=True,
        )
        print(f"✓ Deleted BigQuery dataset and tables: {PROJECT_ID}.{DATASET_ID}")
    except (NotFound, ResourceExhausted) as exc:
        print(f"BigQuery dataset already removed: {exc}")